In [1]:
import sys
import os

os.chdir("..")
print("Working directory:", os.getcwd())

sys.path.append(".")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import warnings
warnings.filterwarnings("ignore")

nltk.download("vader_lexicon", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)

from nltk.sentiment.vader import SentimentIntensityAnalyzer

from src.load_data import load_clean
from src.sentiment import run_sentiment


Working directory: C:\Users\Jessica\Desktop\projects\tiktok-analysis-dashboard


In [2]:
df = load_clean()
print(f"Shape: {df.shape}")
df.head()

Loaded clean data: (658, 15)
Shape: (658, 15)


,id_video,duration_seconds,text_part,hashtags,human_time,author_followers,author_likes,id_sound,views,likes,engagement_rate,post_date,post_hour,post_month,creator_tier
0,1,13.0,А че такое?,"попугай, жако, попугайжако, говорящийпопугай, ...",2018-08-08 20:58:31,675900.0,7500000.0,7.028884e+18,144600000.0,4400000.0,0.030429,2018-08-08,20,8,high followers
1,2,9.0,Таксист в шоке😱 Inst: Andrey_Pryahin,NaN,2016-01-29 06:16:13,2300000.0,42600000.0,6.870179e+18,113300000.0,5300000.0,0.046778,2016-01-29,6,1,high followers
2,4,60.0,"Учитесь хитрить, как она 😄😉",NaN,2021-03-10 19:57:42,523900.0,11200000.0,6.984582e+18,96700000.0,3000000.0,0.031024,2021-03-10,19,3,high followers
3,5,16.0,телефонный пранк от говорящего ворона Карлушы,"пранк, шутка, говорящийворонкарлуша, воронгово...",2020-03-21 11:59:19,943700.0,16800000.0,7.052249e+18,139200000.0,5900000.0,0.042385,2020-03-21,11,3,high followers
4,6,21.0,Statement ✍️,"tennis, tennistv, atptour, fritz",2018-12-17 19:36:27,1800000.0,138300000.0,7.458512e+18,35800.0,1822.0,0.050894,2018-12-17,19,12,high followers


In [3]:
#sentiment distribution
df["sentiment"].hist(bins=50, figsize=(8, 4), color="#7F77DD")
plt.axvline(0, color="red", linestyle="--", linewidth=1)
plt.title("Distribution of sentiment scores")
plt.xlabel("Compount sentiment score")
plt.ylabel("Count")
plt.show()

KeyError: 'sentiment'

In [ ]:
#pie breakdown of sentijment
df["sentiment_label"].value_counts().plot(
    kind="pie", autopct="%1.1f%%", figsize=(6, 6),
    colors=["#1D9E75", "#888780", "#E24B4A"]
)
plt.title("Sentiment breakdown of captions")
plt.ylabel("")
plt.show()

In [ ]:
#visualize setiment affects views
df.groupby("sentiment_label")["views"].mean().sort_values().plot(
    kind="barh", figsize=(8, 4), 
    color=["#E24B4A", "#888780", "#1D9E75"]
)
plt.title("Average Views by Sentiment Label")
plt.xlabel("Average Views")
plt.show()

In [ ]:
#visualize sentiment affects likes
df.groupby("sentiment_label")["likes"].mean().sort_values().plot(
    kind="barh", figsize=(8, 4), 
    color=["#E24B4A", "#888780", "#1D9E75"]
)
plt.title("Average Likes by Sentiment Label")
plt.xlabel("Average Likes")
plt.show()

In [ ]:
# avg engagement rate by sentiment label
df.groupby("sentiment_label")["engagement_rate"].mean().sort_values().plot(
    kind="barh", figsize=(8, 4),
    color=["#E24B4A", "#888780", "#1D9E75"]
)
plt.title("Average engagement rate by sentiment label")
plt.xlabel("Avg engagement rate (likes / views)")
plt.show()

In [ ]:
#check correlation of sentiment vs. views 
from scipy import stats

for tier in ["low followers", "high followers"]:
    subset = df[df["creator_tier"] == tier].dropna(subset=["sentiment", "views"])
    r, p = stats.pearsonr(subset["sentiment"], subset["views"])
    print(f"{tier}: r= {r:.3f}, p= {p:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, tier in zip(axes, ["low followers", "high followers"]):
    subset = df[df["creator_tier"] == tier]
    ax.scatter(subset["sentiment"], subset["views"], alpha=0.2, s=8, color="#7F77DD")
    ax.set_title(f"Sentiment vs. Views - {tier}")
    ax.set_xlabel("Sentiment score")
    ax.set_ylabel("Views")

plt.tight_layout()
plt.show()

In [ ]:
print("---- Most Positive Captions ----")
print(df.nlargest(5, "sentiment")[["text_part", "sentiment", "views"]].to_string())

print("\n---- Most Negative Captions ----")
print(df.nsmallest(5, "sentiment")[["text_part", "sentiment", "views"]].to_string())

In [ ]:
df.to_csv("data/processed/tiktok_sentiment.csv", index=False)
print("Saved to data/processed/tiktok_sentiment.csv")

In [ ]:
## SENTIMENT ANALYSIS
#- Breakdown:** Percentage of captions are positive/neutral/negative
#- Engagement:** sentiment label gets movt views/likes
#- Correlation:** r value is positive/negative and if p < 0.05
#- Creator Level:** analyzes if sentiment matters more for small/large creators
#- Conclusion:** When analyzing caption sentiment, does it predict the engagement?